# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all data structures by their `@id` fields according to the Croissant schema.

### Dataset Source
The dataset metadata and structure are described via a Croissant schema located at the following URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and columns in the dataset.

In [ ]:
# List all record sets and their fields according to their '@id'
record_sets = list(dataset.record_sets.keys())
print("Available record sets (by @id):")
for rs_id in record_sets:
    print(f"  - {rs_id}")

# Show fields (columns) within each record set by their @id
record_set_fields = {}

for rs_id in record_sets:
    rs_obj = dataset.record_sets[rs_id]
    print(f"\nFields in record set '@id': {rs_id}")
    fields = [field['@id'] if isinstance(field, dict) and '@id' in field else field for field in getattr(rs_obj, 'field', [])]
    record_set_fields[rs_id] = fields
    for field_id in fields:
        print(f"    - {field_id}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame, referencing record set and field `@id`s.

In [ ]:
# Extract data from each record set using their @id
dfs = {}
for rs_id in record_sets:
    print(f"Loading records for record set '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"    Loaded {df.shape[0]} records with columns: {df.columns.tolist()}")

# Preview one of the main record sets
# Example: use the first record set by @id
main_record_set_id = record_sets[0]
print(f"\nColumns for record set '@id': {main_record_set_id}")
print(dfs[main_record_set_id].columns.tolist())

dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Process and explore the data using filtering, normalization, and grouping. All columns and fields are referenced by their `@id`.

In [ ]:
# In this section, select a numeric field by @id for EDA
# For illustration, if there is an age field or similar numeric field, use its @id.
# Otherwise, choose any numeric field available in the dataset.

# -- Define these IDs based on actual field @ids discovered above --
record_set_id = main_record_set_id  # Use the main record set
df = dfs[record_set_id]

print(f"\nColumns in main record set '{record_set_id}':\n{df.columns.tolist()}")

numeric_field_candidates = [c for c in df.select_dtypes(include='number').columns if c != '']
if len(numeric_field_candidates) == 0:
    print("No numeric fields found in this dataset for EDA.")
else:
    numeric_field = numeric_field_candidates[0]  # Select the first available numeric field
    print(f"Using numeric field '@id': {numeric_field}")

    # Filter for records above a threshold (arbitrarily chosen as median)
    threshold = df[numeric_field].median()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

    # Try grouping by a categorical field (choose a non-numeric field)
    group_field_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == 'object']
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field}, aggregated mean {numeric_field}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships using the fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple visualization for the main numeric field
if len(numeric_field_candidates) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouping field was found, show grouped means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        grouped_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        sns.barplot(x=grouped_means.index, y=grouped_means.values)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, inspect, and analyze the FAIR^2 tabular dataset of second primary colorectal cancer in cancer survivors. All dataset entities and fields were referenced dynamically by their `@id`, enabling robust and schema-compliant data processing. Further in-depth biomedical or statistical analyses are now possible using these prepared DataFrames.